
Inference notebook:
- https://www.kaggle.com/code/valentinwerner/893-deberta3base-inference

## 🏟️ Credits (because this baseline did mostly already exist when I joiend)

- @Nicholas Broad published the transformer baseline which performs only marginally worse: https://www.kaggle.com/code/nbroad/transformer-ner-baseline-lb-0-854
- @Joseph Josia published the training notebook which I basically copy pasted (which is based itself on nbroad, but yeah): https://www.kaggle.com/code/takanashihumbert/piidd-deberta-model-starter-training



## 💡 What I added
- Downsampling negative samples (samples without labels, but they possible still work as examples where names should not be tagged as name)
- Adding @moths external data: https://www.kaggle.com/competitions/pii-detection-removal-from-educational-data/discussion/469493
- Adding PJMathematicianss external data: https://www.kaggle.com/competitions/pii-detection-removal-from-educational-data/discussion/470921
- However, I used my cleaned version instead (the punctuation is flawed in the original data set at the time of this trainign): https://www.kaggle.com/code/valentinwerner/fix-punctuation-tokenization-external-dataset

Doing this brought the LB score to .888 - Trained in Kaggle Notebook, no tricks or secrets.

- I added emojis because that seems to be the kaggle upvote meta

## 📝 Config & Imports
- 1024 max length has been working well for me. As some samples are longer, you may want to go as high as you can 

In [ ]:
TRAINING_MODEL_PATH = "microsoft/deberta-v3-base"
TRAINING_MAX_LENGTH = 1024
OUTPUT_DIR = "output"

In [ ]:
!pip install seqeval evaluate -q

In [ ]:
import json
import pandas as pd
import argparse
from itertools import chain
from functools import partial

import torch
from transformers import AutoTokenizer, Trainer, TrainingArguments
from transformers import AutoModelForTokenClassification, DataCollatorForTokenClassification
import evaluate
from datasets import Dataset, features
import numpy as np

# CV Split

In [ ]:
data = json.load(open("/kaggle/input/pii-detection-removal-from-educational-data/train.json"))
df = pd.DataFrame(data)

df.shape

In [ ]:
def encode_labels(df):
    df["unique_labels"] = df["labels"].apply(lambda x: list(set(
        [l.split('-')[1] for l in x if l != 'O']
         )))
    # add 1-hot encoding
    from sklearn.preprocessing import MultiLabelBinarizer

    mlb = MultiLabelBinarizer()
    one_hot_encoded = mlb.fit_transform(df['unique_labels'])
    one_hot_df = pd.DataFrame(one_hot_encoded, columns=mlb.classes_)
    df = pd.concat([df, one_hot_df], axis=1)
    
    # add 'OTHER' column
    df['OTHER'] = df['unique_labels'].apply(lambda x: 1 if len(x) == 0 else 0)
    
    return df, list(mlb.classes_) + ['OTHER']

df, label_classes = encode_labels(df)

for col in label_classes:
    print(f'{col}: {df[col].sum()}')

In [ ]:
# Shuffle the dataframe
df = df.sample(frac=1, random_state=42)
df = df.reset_index(drop=False, names="orig_index")

# Create a 'valid' column and set it to False
df['valid'] = False

# Define the validation numbers
val_nums = {
    'EMAIL': 12,
    'ID_NUM': 12,
    'NAME_STUDENT': 100,
    'PHONE_NUM': 4,
    'STREET_ADDRESS': 2,
    'URL_PERSONAL': 20,
    'USERNAME': 5,
    'OTHER': 1000, 
}

# For each class in val_nums, randomly select the specified number of examples and set 'valid' to True
for label, num in val_nums.items():
    valid_indices = df[df[label] == 1].sample(n=num, random_state=42).index
    df.loc[valid_indices, 'valid'] = True


# Let's double check the classes per split:
for col in label_classes:
    print(f'VALID {col}: {df[df.valid == True][col].sum()}')
    print(f'TRAIN {col}: {df[df.valid == False][col].sum()}')

In [ ]:
df['valid'].value_counts()

In [ ]:
# data[0]

In [ ]:
train_data = []
valid_data = []

# Iterate over the DataFrame using itertuples for efficiency
for row in df.itertuples():
    # Access data using row.Index to find the corresponding item in the 'data' list
    if row.valid:  # If valid column is True
        valid_data.append(data[row.Index])
    else:
        train_data.append(data[row.Index])
        
print("Train data: ", len(train_data))
print("Valid data: ", len(valid_data))

## 🗺️ Data Selection and Label Mapping
- As mentioned before, I additionaly use the moth dataset

In [ ]:
# downsampling of negative examples
p=[] # positive samples (contain relevant labels)
n=[] # negative samples (presumably contain entities that are possibly wrongly classified as entity)
for d in train_data:
    if any(np.array(d["labels"]) != "O"): p.append(d)
    else: n.append(d)
print("original datapoints: ", len(train_data))

external = json.load(open("/kaggle/input/fix-punctuation-tokenization-external-dataset/pii_dataset_fixed.json"))
print("external datapoints: ", len(external))

moredata = json.load(open("/kaggle/input/fix-punctuation-tokenization-external-dataset/moredata_dataset_fixed.json"))
print("moredata datapoints: ", len(moredata))

nbroaddata = json.load(open("/kaggle/input/pii-dd-mistral-generated/mixtral-8x7b-v1.json"))
print("nbroaddata datapoints: ", len(nbroaddata))

data = nbroaddata+moredata+external+p+n[:len(n)//3]
print("combined: ", len(data))

In [ ]:
all_labels = sorted(list(set(chain(*[x["labels"] for x in data]))))
label2id = {l: i for i,l in enumerate(all_labels)}
id2label = {v:k for k,v in label2id.items()}

target = [
    'B-EMAIL', 'B-ID_NUM', 'B-NAME_STUDENT', 'B-PHONE_NUM', 
    'B-STREET_ADDRESS', 'B-URL_PERSONAL', 'B-USERNAME', 'I-ID_NUM', 
    'I-NAME_STUDENT', 'I-PHONE_NUM', 'I-STREET_ADDRESS', 'I-URL_PERSONAL'
]

print(id2label)

## ♟️ Data Tokenization
- This tokenizer is actually special, comparing to usual NLP challenges

In [ ]:
def tokenize(example, tokenizer, label2id, max_length):

    # rebuild text from tokens
    text = []
    labels = []

    for t, l, ws in zip(
        example["tokens"], example["provided_labels"], example["trailing_whitespace"]
    ):
        text.append(t)
        labels.extend([l] * len(t))

        if ws:
            text.append(" ")
            labels.append("O")

    # actual tokenization
    tokenized = tokenizer("".join(text), return_offsets_mapping=True, max_length=max_length)

    labels = np.array(labels)

    text = "".join(text)
    token_labels = []

    for start_idx, end_idx in tokenized.offset_mapping:
        # CLS token
        if start_idx == 0 and end_idx == 0:
            token_labels.append(label2id["O"])
            continue

        # case when token starts with whitespace
        if text[start_idx].isspace():
            start_idx += 1

        token_labels.append(label2id[labels[start_idx]])

    length = len(tokenized.input_ids)

    return {**tokenized, "labels": token_labels, "length": length}

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(TRAINING_MODEL_PATH)

ds = Dataset.from_dict({
    "full_text": [x["full_text"] for x in data],
    "document": [str(x["document"]) for x in data],
    "tokens": [x["tokens"] for x in data],
    "trailing_whitespace": [x["trailing_whitespace"] for x in data],
    "provided_labels": [x["labels"] for x in data],
})

valid_ds = Dataset.from_dict({
    "full_text": [x["full_text"] for x in valid_data],
    "document": [str(x["document"]) for x in valid_data],
    "tokens": [x["tokens"] for x in valid_data],
    "trailing_whitespace": [x["trailing_whitespace"] for x in valid_data],
    "provided_labels": [x["labels"] for x in valid_data],
})


ds = ds.map(tokenize, fn_kwargs={"tokenizer": tokenizer, "label2id": label2id, "max_length": TRAINING_MAX_LENGTH}, num_proc=3)
valid_ds = valid_ds.map(tokenize, fn_kwargs={"tokenizer": tokenizer, "label2id": label2id, "max_length": TRAINING_MAX_LENGTH}, num_proc=3)
# ds = ds.class_encode_column("group")

In [ ]:
# x = ds[0]

# for t,l in zip(x["tokens"], x["provided_labels"]):
#     if l != "O":
#         print((t,l))

# print("*"*100)

# for t, l in zip(tokenizer.convert_ids_to_tokens(x["input_ids"]), x["labels"]):
#     if id2label[l] != "O":
#         print((t,id2label[l]))

## 🧮 Competition metrics
- Note that we are not using the normal F1 score.
- Although it is early in the competition, there are plenty of discsussions already explaining this:
- e.g., here: https://www.kaggle.com/competitions/pii-detection-removal-from-educational-data/discussion/470024

In [ ]:
from seqeval.metrics import recall_score, precision_score
from seqeval.metrics import classification_report
from seqeval.metrics import f1_score

def compute_metrics(p, all_labels):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # Remove ignored index (special tokens)
    true_predictions = [
        [all_labels[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [all_labels[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    
    recall = recall_score(true_labels, true_predictions)
    precision = precision_score(true_labels, true_predictions)
    f1_score = (1 + 5*5) * recall * precision / (5*5*precision + recall)
    
    results = {
        'recall': recall,
        'precision': precision,
        'f1': f1_score
    }
    return results

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    TRAINING_MODEL_PATH,
    num_labels=len(all_labels),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)
collator = DataCollatorForTokenClassification(tokenizer, pad_to_multiple_of=16)

In [ ]:
# I decided to uses no eval
# final_ds = ds.train_test_split(test_size=0.2, seed=42) # cannot use stratify_by_column='group'
# final_ds

# Wandb setup

In [ ]:
from kaggle_secrets import UserSecretsClient
import wandb

user_secrets = UserSecretsClient()
wandb_api_key = user_secrets.get_secret("wandb")
wandb.login(key=wandb_api_key)
wandb.init(project='kaggle_pii_detect', job_type='preprocessing')

## 🏋🏻‍♀️ Training
- I actually do not use an eval set for submission to train on all data
- Values are not really tuned and go by gut feeling, as this is my first iteration / baseline

In [ ]:
# I actually chose to not use any validation set. This is only for the model I use for submission.
args = TrainingArguments(
    output_dir=OUTPUT_DIR, 
    fp16=True,
    learning_rate=2e-5,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    report_to="wandb",
    evaluation_strategy="steps",
    do_eval=True,
    save_total_limit=3,
    logging_steps=50,
    eval_steps=50,
    lr_scheduler_type='cosine',
    metric_for_best_model="f1",
    greater_is_better=True,
    warmup_ratio=0.1,
    weight_decay=0.01,
    seed = 42
)

trainer = Trainer(
    model=model, 
    args=args, 
    train_dataset=ds,
    eval_dataset=valid_ds,
    data_collator=collator, 
    tokenizer=tokenizer,
    compute_metrics=partial(compute_metrics, all_labels=all_labels),
)

In [ ]:
%%time
trainer.train()

## 💾 Save models
- You can click on "Save version" (top right) and "Save & Run All (Commit)"
- Then you can use this notebook as input for your inference notebook

In [ ]:
trainer.save_model("deberta3base_1024")
tokenizer.save_pretrained("deberta3base_1024")